# Predict with CNN Model (PyTorch) for Land Use Classification
Dự đoán sử dụng đất bằng mô hình CNN đã huấn luyện

In [ ]:
%%time
%matplotlib inline

import importlib
import new_import_ODC  

importlib.reload(new_import_ODC)

from new_import_ODC import *

In [ ]:
# Kiểm tra GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    device = 'cuda'
else:
    print("Using CPU for inference")
    device = 'cpu'

print(f"\nDevice sẽ dùng: {device}")

In [ ]:
%%time
# Cấu hình Daskgateway
cluster, client = notebook_utils.initialize_dask(use_gateway=True, workers=(1, 10))
# Khai báo 1 Datacube là dc
dc = datacube.Datacube()

# Cấu hình truy cập dịch vụ S3
configure_s3_access(aws_unsigned=False, requester_pays=True, client=client)

client

In [ ]:
## cấu hình thời gian lấy ảnh và tọa độ
date_range = ("2022-09-01", "2023-10-01")
longtitude_range = (105.5, 106.4)
latitude_range = (9.2, 10.0)

coordinates = (longtitude_range, latitude_range)

In [ ]:
## truy vấn ảnh vệ tinh sen2
data = load_data(dc, date_range, longtitude_range, latitude_range)
notebook_utils.heading(notebook_utils.xarray_object_size(data))
display(data)

In [ ]:
%%time
# Tiến hành loại bỏ các vị trí bị mây ảnh hưởng
result = mask_clean(data)
progress(result)

In [ ]:
# Tiến hành tính toán NDVI
ds1 = calculate_indices(result, index="NDVI", satellite_mission="s2")
ndvi = ds1["NDVI"]
display(ndvi)

In [ ]:
# Thiết lập giá trị trung bình mùa vụ để xử lý các điểm ảnh bị mây dựa vào sự thay đổi theo mùa
time_split = [
    slice("2022-09-01", "2023-01-01"),
    slice("2023-01-01", "2023-05-01"),
    slice("2023-05-01", "2023-07-01"),
    slice("2023-07-01", "2023-10-01"),
]

# Điền mây ở các vị trí mang giá trị nan (fill nan)
fill_nan_ndvi = fill_nan(ndvi, time_split)

In [ ]:
%%time
## tính ndvi theo tháng
average_ndvi = fill_nan_ndvi.resample(time="1M").mean().persist()
progress(average_ndvi)

# compute average_ndvi
average_ndvi = average_ndvi.compute()

In [ ]:
#Load dữ liệu ảnh Sentinel 1
dsvh, dsvv = load_data_sen1(dc, date_range, coordinates)
average_vv = calculate_average(dsvv, time_pattern='1M')
average_vh = calculate_average(dsvh, time_pattern='1M')

## Tải model đã huấn luyện

In [ ]:
# Tải model CNN PyTorch
model, scaler = load_pytorch_model(model_name="model_cnn_pytorch.pth", device=device)
model.eval()
print(f"\n✅ Model đã được tải thành công")

## Dự đoán cho toàn bộ khu vực

In [ ]:
%%time
# Chuẩn bị dữ liệu dự đoán
print("🔷 Chuẩn bị dữ liệu dự đoán...")

# Lấy kích thước của ảnh
num_y = average_ndvi.shape[1]
num_x = average_ndvi.shape[2]

print(f"Kích thước ảnh: {num_y} x {num_x}")

# Chuẩn bị dữ liệu dự đoán
predictions = []

print(f"\n🔍 Dự đoán từng pixel...")
batch_size = 128

with torch.no_grad():
    for y_idx in range(num_y):
        y_predictions = []
        
        # Lấy dữ liệu cho từng hàng (row)
        ndvi_row = average_ndvi.isel(y=y_idx).values  # shape: (time, x)
        vh_row = average_vh.sel(y=average_ndvi.y.values[y_idx], method='nearest').values  # shape: (time, x)
        vv_row = average_vv.sel(y=average_ndvi.y.values[y_idx], method='nearest').values  # shape: (time, x)
        
        # Xử lý theo batch
        for x_idx in range(0, num_x, batch_size):
            x_end = min(x_idx + batch_size, num_x)
            batch_size_actual = x_end - x_idx
            
            # Tạo batch data
            batch_data = np.zeros((batch_size_actual, ndvi_row.shape[0] * 3))
            
            for idx, x_i in enumerate(range(x_idx, x_end)):
                ndvi_data = ndvi_row[:, x_i]
                vh_data = vh_row[:, x_i]
                vv_data = vv_row[:, x_i]
                batch_data[idx, :] = np.concatenate((ndvi_data, vh_data, vv_data))
            
            # Normalize dữ liệu
            batch_data_scaled = scaler.transform(batch_data)
            batch_data_reshaped = batch_data_scaled.reshape(batch_size_actual, 1, -1)
            
            # Convert to tensor
            batch_tensor = torch.FloatTensor(batch_data_reshaped).to(device)
            
            # Dự đoán
            outputs = model(batch_tensor)
            _, predicted = torch.max(outputs.data, 1)
            
            y_predictions.extend(predicted.cpu().numpy().tolist())
        
        predictions.extend(y_predictions)
        
        if (y_idx + 1) % 100 == 0:
            print(f"  Đã xử lý {y_idx + 1}/{num_y} hàng...")

# Reshape predictions
predictions = np.array(predictions).reshape(num_y, num_x)
print(f"\n✅ Hoàn thành dự đoán!")
print(f"   Shape: {predictions.shape}")

In [ ]:
# Tạo xarray DataArray từ predictions
final_label = predictions
final_xarray_save = xr.DataArray(final_label, dims=("y", "x"))
final_xarray_save = final_xarray_save.rio.write_crs(average_ndvi.rio.crs)

x_values = average_ndvi.x.values
y_values = average_ndvi.y.values

data_array = xr.DataArray(final_xarray_save,
                          coords={'x': x_values, 'y': y_values},
                          dims=['y', 'x'])
data_array = data_array.rio.write_crs(average_ndvi.rio.crs)

print(f"✅ DataArray tạo thành công")
print(f"   Shape: {data_array.shape}")
print(f"   CRS: {data_array.rio.crs}")

In [ ]:
# Hiển thị kết quả dự đoán
fig, ax = plt.subplots(figsize=(12, 10))

# Cấu hình colormap
cmap = plt.cm.get_cmap('tab10')
im = ax.imshow(data_array.values, cmap=cmap, interpolation='nearest')

# Tạo colorbar
cbar = plt.colorbar(im, ax=ax, label='Land Use Class')
cbar.set_ticks([0, 1, 2, 3, 4, 5, 6, 7])
cbar.set_ticklabels(['Lua tom', 'Lua', 'CHN', 'CLN', 'TS', 'Song', 'Dat xay dung', 'Rung'])

ax.set_title('Land Use Classification Map (CNN PyTorch)', fontsize=14, fontweight='bold')
ax.set_xlabel('X coordinate')
ax.set_ylabel('Y coordinate')

plt.tight_layout()
plt.show()

In [ ]:
# Lưu kết quả dự đoán
output_path = "prediction_results/classification_map_cnn_pytorch.tif"
os.makedirs("prediction_results", exist_ok=True)

data_array.rio.to_raster(output_path)
print(f"✅ Kết quả đã lưu tại: {output_path}")

In [ ]:
# đóng client, cluster
client.close()
cluster.close()